In [ ]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("willianoliveiragibin/global-science-data")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

files = os.listdir(path)
print('file yang tersedia:', files)

csv_file = os.path.join(path, files[0])
df=pd.read_csv(csv_file)
pd.set_option('display.max_column', None)

In [ ]:
df.isnull().sum()
print(df.info())
print(df.columns.to_list())
display(df.describe())
display(df.head())

df=df.rename(columns={
    '国家' : 'country',
    '活跃仓储数量': 'active_storage_count',
    '获得质量认证的仓储数量' : 'certified_storage_count',
    '质量认证率' : 'certification_rate'
})
print(df.columns.to_list())
display(df.head())
print(df['country'].unique())
print(df['country'].value_counts())

In [ ]:
top_country = df.groupby('country')['active_storage_count'].sum().sort_values(ascending=False).head()
nama_country = {
    '美国': 'United States',
    '德国': 'Germany',
    '加拿大': 'Canada',
    '英国': 'United Kingdom',
    '欧盟': 'European Union'
}
df['country'] = df['country'].replace(nama_country)
print('top 5 negara dengan gudang aktif terbanyak:\n', top_country)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
top_country.plot(kind='bar', color='skyblue')
plt.title('Top 5 Negara dengan \n Jumlah Gudang Aktif Terbanyak', fontsize=12, color='black')
plt.xlabel('country', fontsize=11)
plt.ylabel('count', fontsize=11)
plt.xticks(rotation=45)
plt.ylim(ymin=0, ymax=1100)
plt.tight_layout()
plt.show()

In [ ]:
# df['certification_rate'] = df['certification_rate'].str.replace('%','').astype(float)
korelasi= df[['certified_storage_count', 'certification_rate']].corr()
print('matrix korelasi:\n', korelasi)

plt.figure()
sns.scatterplot(data=korelasi, x='certified_storage_count', y='certification_rate', color='skyblue')
plt.title('Korelasi antara Jumlah Gudang Bersertifikasi\n dengan Presentase Sertifikasi', fontsize=12, color='black')
plt.xlabel('jumlah gudang')
plt.ylabel('rate')
plt.grid(color='darkgray', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

a = LabelEncoder()
df['re3data_subject']= a.fit_transform(df['re3data_subject'].astype('str'))

x = df[['re3data_subject', 'active_storage_count']]
y= df['certification_rate']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=42)

model=LinearRegression()
model.fit(x_train,y_train)

y_pred = model.predict(x_test)
print('metrics r2:', r2_score(y_test,y_pred))
print('metrics eror:', mean_squared_error(y_test,y_pred))
